# Notebook 2: `node1` Cluster Bootstrap

Run this notebook **from inside `node1`** after Terraform provisioning is complete and this repository has been cloned locally on `node1`.

Execution order in this notebook:
1. Ansible connectivity check
2. Pre-K8s node preparation
3. Kubespray `cluster.yml` cluster creation
4. Post-K8s configuration
5. Argo CD application bootstrap

In [ ]:
set -euo pipefail

REPO_ROOT="${REPO_ROOT:-$HOME/recipe-scraper-mlops}"
ANSIBLE_DIR="${ANSIBLE_DIR:-$REPO_ROOT/devops/ansible}"
KUBESPRAY_DIR="${KUBESPRAY_DIR:-$ANSIBLE_DIR/k8s/kubespray}"
KUBESPRAY_INVENTORY="${KUBESPRAY_INVENTORY:-$ANSIBLE_DIR/k8s/inventory/mycluster/hosts.yaml}"

cat <<'EOF'
Parameter summary:
  REPO_ROOT           = $REPO_ROOT
  ANSIBLE_DIR         = $ANSIBLE_DIR
  KUBESPRAY_DIR       = $KUBESPRAY_DIR
  KUBESPRAY_INVENTORY = $KUBESPRAY_INVENTORY
EOF

In [ ]:
set -euo pipefail

HOST_SHORT="$(hostname -s)"
if [[ "$HOST_SHORT" != node1* ]]; then
  echo "This notebook must run on node1. Current host: $HOST_SHORT" >&2
  exit 1
fi

for cmd in ansible-playbook kubectl ssh bash; do
  command -v "$cmd" >/dev/null || {
    echo "Missing required command: $cmd" >&2
    exit 1
  }
done

[[ -d "$REPO_ROOT" ]] || {
  echo "Repository path not found: $REPO_ROOT" >&2
  exit 1
}

for path in \
  "$ANSIBLE_DIR/inventory.yml" \
  "$ANSIBLE_DIR/general/hello_host.yml" \
  "$ANSIBLE_DIR/pre_k8s/pre_k8s_configure.yml" \
  "$ANSIBLE_DIR/post_k8s/post_k8s_configure.yml" \
  "$ANSIBLE_DIR/argocd/argocd_bootstrap_apps.yml" \
  "$KUBESPRAY_INVENTORY"; do
  [[ -f "$path" ]] || {
    echo "Missing expected file: $path" >&2
    exit 1
  }
done

if [[ ! -f "$KUBESPRAY_DIR/cluster.yml" ]]; then
  echo "Missing Kubespray playbook: $KUBESPRAY_DIR/cluster.yml" >&2
  echo "Initialize submodule if needed: git submodule update --init --recursive" >&2
  exit 1
fi

echo "Prerequisites passed on node1."

## Inventory Drift Checkpoint

Before running playbooks, confirm host IPs in `devops/ansible/inventory.yml` and `devops/ansible/k8s/inventory/mycluster/hosts.yaml` still match Terraform outputs.

## Phase 1: Ansible Connectivity

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/general/hello_host.yml

## Phase 2: Pre-K8s Preparation

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/pre_k8s/pre_k8s_configure.yml

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible all -i devops/ansible/inventory.yml -m shell -a 'ip -brief address && ip route' -o

## Phase 3: Kubespray Cluster Creation

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i "$KUBESPRAY_INVENTORY" "$KUBESPRAY_DIR/cluster.yml" -u cc -b

In [ ]:
set -euo pipefail

if [[ -f "$HOME/.kube/config" ]]; then
  kubectl get nodes -o wide
  kubectl get pods -A
else
  sudo kubectl --kubeconfig /etc/kubernetes/admin.conf get nodes -o wide
  sudo kubectl --kubeconfig /etc/kubernetes/admin.conf get pods -A
fi

## Phase 4: Post-K8s Configuration

Note: this playbook prints the initial Argo CD admin password in output.

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook -i devops/ansible/inventory.yml devops/ansible/post_k8s/post_k8s_configure.yml

In [ ]:
set -euo pipefail

kubectl get nodes -o wide
argocd version --client
argo version --client
kubectl get all -n argocd

## Phase 5: Argo CD Bootstrap Apps

In [ ]:
set -euo pipefail

cd "$REPO_ROOT"
ansible-playbook devops/ansible/argocd/argocd_bootstrap_apps.yml

In [ ]:
set -euo pipefail

kubectl get applications -n argocd
kubectl get pods -n argocd
echo "Optional if argocd CLI login already configured: argocd app list && argocd app get platform"

## Recovery Notes (Reruns)

- If a phase fails, fix root cause and rerun only that phase cell (and its validation cell).
- If Kubespray fails mid-run, rerun the same `cluster.yml` command after confirming inventory/SSH reachability.
- If Argo CD apps partially apply, rerun `argocd_bootstrap_apps.yml`; Kubernetes apply is idempotent for unchanged manifests.

## Final Verification + Next Checks

Confirm:
- Cluster nodes are `Ready`.
- Argo CD Applications are registered.
- Core workloads converge to healthy state.

Useful watch commands:
- `kubectl get pods -A`
- `kubectl get applications -n argocd -w`

Security note:
- If this notebook output will be shared, clear outputs first because post-K8s phase may include sensitive bootstrap credentials.